# ReLU Tessellation Dynamics: Standard vs Adversarial Training

This notebook runs the full experimental pipeline on Google Colab with GPU support.

**Requirements:** Select a **GPU runtime** before running (Runtime > Change runtime type > T4 GPU).

The setup installs:
- `graph-tool` via conda (required by SplineCam for exact tessellation computation)
- SplineCam from the official repository
- All other Python dependencies (torch, numpy, scipy, matplotlib, etc.)

## 0. Verify GPU availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
        "then restart the runtime."
    )

## 1. Install graph-tool via conda (matched to kernel Python)

`graph-tool` is a C++ library not available via pip. The Ubuntu apt package is built for Python 3.10, but Colab's kernel runs Python 3.12 — the compiled extensions are ABI-incompatible across minor versions.

We install miniconda and use conda-forge to get a graph-tool build that matches the kernel's exact Python version. This takes 3-5 minutes.

In [ ]:
import sys
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
print(f"Colab kernel Python: {PY_VER} ({sys.executable})")
print(f"We will install graph-tool for Python {PY_VER} via conda-forge.")

## 2. Install miniconda, graph-tool, and fix system libraries

The next two cells handle:
1. **Cell A**: Installs miniconda + graph-tool via conda-forge (skip if already done)
2. **Cell B**: Copies the conda env's newer `libstdc++` over the system version and **restarts the runtime** (only on first run — it detects if already done)
3. **Cell C**: Adds the conda env to `sys.path` and imports `graph_tool`

After the restart, re-run all cells from the top. On second run, Cell B will skip the restart and Cell C will import normally.

In [ ]:
import os, sys

CONDA_PREFIX = "/opt/conda"
CONDA = f"{CONDA_PREFIX}/bin/conda"
PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"  # Colab kernel version
GT_ENV = f"{CONDA_PREFIX}/envs/gt"

# Step 1: Install miniconda if not present
if not os.path.exists(CONDA):
    print("=== Installing miniconda ===")
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p {CONDA_PREFIX} 2>&1 | tail -2
    !rm /tmp/miniconda.sh

# Step 2: Configure conda — remove defaults entirely, use only conda-forge
!{CONDA} config --remove channels defaults 2>/dev/null; true
!{CONDA} config --add channels conda-forge 2>/dev/null; true
!{CONDA} config --set channel_priority strict
print("Conda channels configured (conda-forge only)")

# Step 3: Create env with --override-channels to fully bypass any default/TOS channels
if not os.path.exists(GT_ENV):
    print(f"\n=== Creating conda env 'gt' with Python {PY_VER} + graph-tool ===")
    print("(This takes 3-5 minutes...)")
    !{CONDA} create -y --override-channels -c conda-forge -n gt python={PY_VER} graph-tool 2>&1 | tail -15
else:
    print(f"Conda env 'gt' already exists at {GT_ENV}")

# Step 4: Verify
print("\n=== Verification ===")
gt_python = f"{GT_ENV}/bin/python"
if os.path.exists(gt_python):
    !{gt_python} --version
    !{gt_python} -c "import graph_tool; print('graph-tool', graph_tool.__version__)"
else:
    print(f"ERROR: {gt_python} not found — env creation may have failed.")
    print("Check the output above for errors.")

In [ ]:
import os, sys, shutil, subprocess

GT_ENV = "/opt/conda/envs/gt"
conda_lib = f"{GT_ENV}/lib"
dst_dir = "/lib/x86_64-linux-gnu"
src = os.path.realpath(f"{conda_lib}/libstdc++.so.6")
dst = f"{dst_dir}/libstdc++.so.6"

# Check if the system libstdc++ already has CXXABI_1.3.15
result = subprocess.run(["strings", dst], capture_output=True, text=True)
if "CXXABI_1.3.15" in result.stdout:
    print("System libstdc++ already has CXXABI_1.3.15 — no restart needed.")
else:
    print(f"Copying {src} -> {dst}")
    shutil.copy2(src, dst)
    # Verify the copy worked
    result2 = subprocess.run(["strings", dst], capture_output=True, text=True)
    assert "CXXABI_1.3.15" in result2.stdout, "Copy failed — CXXABI_1.3.15 still missing"
    print("Library updated. Restarting runtime to load the new version...")
    print(">>> After restart, re-run from Cell 0 (GPU check) through here, then continue. <<<")
    os.kill(os.getpid(), 9)  # Force kernel restart

In [ ]:
import sys, os

PY_VER = f"{sys.version_info.major}.{sys.version_info.minor}"
GT_ENV = "/opt/conda/envs/gt"
conda_sp = f"{GT_ENV}/lib/python{PY_VER}/site-packages"
conda_lib = f"{GT_ENV}/lib"

# Add conda env's site-packages and libraries to the search path
if conda_sp not in sys.path:
    sys.path.insert(0, conda_sp)
ld_path = os.environ.get("LD_LIBRARY_PATH", "")
if conda_lib not in ld_path:
    os.environ["LD_LIBRARY_PATH"] = f"{conda_lib}:{ld_path}"

print(f"Python: {PY_VER}")
print(f"conda site-packages: {conda_sp}")

import graph_tool
print(f"\ngraph-tool version: {graph_tool.__version__}")
print(f"Location: {graph_tool.__file__}")

# Install remaining pip packages into Colab's default Python
!pip install -q torch torchvision matplotlib numpy scipy seaborn tqdm \
    python-igraph networkx livelossplot

import torch
print(f"\nPyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## 3. Clone repositories and install SplineCam

In [ ]:
import os, sys

# Clone SplineCam
SPLINECAM_DIR = "/content/splinecam"
if not os.path.exists(SPLINECAM_DIR):
    !git clone https://github.com/AhmedImtiazPrio/splinecam.git {SPLINECAM_DIR}
else:
    print(f"SplineCam already cloned at {SPLINECAM_DIR}")

# Add to sys.path (repo has no setup.py/pyproject.toml, so pip install -e won't work)
if SPLINECAM_DIR not in sys.path:
    sys.path.insert(0, SPLINECAM_DIR)

# Install SplineCam's dependencies manually
!pip install -q python-igraph networkx

# Verify
import splinecam
print(f"SplineCam imported successfully from {splinecam.__file__}")

In [ ]:
import os

PROJECT_DIR = "/content/adversarial-tesselation-analysis"
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/BryanZhang938/adversarial-tesselation-analysis.git {PROJECT_DIR}
else:
    print(f"Project already cloned at {PROJECT_DIR}")

assert os.path.exists(PROJECT_DIR), f"Project not found at {PROJECT_DIR}"
print(f"Project directory: {PROJECT_DIR}")
print(os.listdir(PROJECT_DIR))

## 4. Patch SplineCam for Colab compatibility

SplineCam has two known issues in Colab:
1. **TorchScript float typing**: `utils.py` uses integer defaults (e.g. `pad_dist=1`) which TorchScript rejects. We patch them to floats.
2. **Multiprocessing + CUDA**: `compute.py` spawns workers that try to initialize CUDA, causing errors. We patch DataLoader `num_workers` to 0.

In [ ]:
import importlib

# --- Patch 1: Fix TorchScript float type inference in utils.py ---
utils_path = os.path.join(SPLINECAM_DIR, "splinecam", "utils.py")
with open(utils_path, "r") as f:
    content = f.read()

patches_applied = []
for old, new in [
    ("pad_dist=1,", "pad_dist: float=1.0,"),
    ("pad_dist=1)", "pad_dist: float=1.0)"),
    ("pad_dist = 1,", "pad_dist: float = 1.0,"),
    ("pad_dist = 1)", "pad_dist: float = 1.0)"),
    ("seed: int = -1", "seed: int = -1"),  # already correct, skip
]:
    if old in content and old != new:
        content = content.replace(old, new)
        patches_applied.append(f"  {old!r} -> {new!r}")

if patches_applied:
    with open(utils_path, "w") as f:
        f.write(content)
    print("Patched utils.py:")
    for p in patches_applied:
        print(p)
else:
    print("utils.py: no patches needed (already correct)")

# --- Patch 2: Disable multiprocessing in compute.py ---
compute_path = os.path.join(SPLINECAM_DIR, "splinecam", "compute.py")
with open(compute_path, "r") as f:
    content = f.read()

# Force num_workers=0 in DataLoader calls to avoid CUDA fork issues
if "num_workers=n_workers" in content:
    content = content.replace("num_workers=n_workers", "num_workers=0")
    with open(compute_path, "w") as f:
        f.write(content)
    print("Patched compute.py: set DataLoader num_workers=0")
elif "num_workers=0" in content:
    print("compute.py: already patched")
else:
    print("compute.py: no DataLoader num_workers found (check manually if errors occur)")

# Reload modules to pick up patches
import splinecam.utils
import splinecam.compute
importlib.reload(splinecam.utils)
importlib.reload(splinecam.compute)
print("\nSplineCam modules reloaded.")

## 5. Verify SplineCam works end-to-end

Quick smoke test: wrap a tiny MLP and compute partitions on a small domain.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from splinecam.wrappers import model_wrapper
from splinecam.compute import get_partitions_with_db

# Build a tiny test model
test_model = nn.Sequential(
    nn.Linear(2, 10),
    nn.ReLU(),
    nn.Linear(10, 2),
).eval().cuda()

# Define square domain
domain = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=np.float64)
domain_t = torch.from_numpy(domain).cuda()

# Wrap model (T=None for 2D input, no projection needed)
NN = model_wrapper(test_model, input_shape=(2,), T=None,
                   dtype=torch.float64, device='cuda')

# Compute tessellation
# For 2D, get_partitions_with_db expects:
#   domain: numpy array of vertices
#   T: identity-like projection (we pass None and handle below)
#   NN: the wrapped model
T_proj = torch.eye(3, dtype=torch.float64, device='cuda')[:2]  # 2x3: [I | 0]
regions, endpoints, Abw = get_partitions_with_db(
    domain_t, T_proj, NN,
    fwd_batch_size=512, batch_size=64, n_workers=0
)

print(f"Tessellation computed successfully!")
print(f"  Number of regions: {len(regions)}")
print(f"  Endpoints type: {type(endpoints)}")
del test_model, NN, regions, endpoints, Abw
torch.cuda.empty_cache()

## 6. Configure and run experiment

Set up the experiment parameters matching the paper methodology:
- Datasets: spirals (T=500) or concentric_rings (T=300)
- Architecture: ReLU MLP, 1-2 hidden layers, width in {10, 20, 50}
- Optimizer: SGD, lr=0.01, momentum=0.9
- Adversarial: PGD-AT with l2 norm, K=7 steps, alpha=epsilon/4
- Tessellation: M=50 evenly spaced checkpoints, SplineCam over [-1.5, 1.5]^2

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

import copy
import json
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from configs.experiment_config import (
    ExperimentConfig, _compute_checkpoint_epochs,
)
from src.datasets import get_dataset, get_dataloader
from src.models import make_relu_mlp, count_parameters
from src.train import train_model
from src.tessellation_analysis import analyze_checkpoint
from src.visualization import (
    plot_dataset, plot_training_comparison,
    plot_epoch_snapshots, plot_boundary_distance_histograms,
)

print("All modules imported successfully.")

In [ ]:
# =====================================================
# EXPERIMENT CONFIGURATION — edit these as needed
# =====================================================
DATASET = "spirals"           # "spirals" or "concentric_rings"
HIDDEN_DIMS = [50]            # e.g. [50], [20, 20], [10]
EPSILON = 0.1                 # l2 perturbation budget
SEED = 42

# Auto-set epochs per paper
EPOCHS = 500 if DATASET == "spirals" else 300

# Build config
config = ExperimentConfig()
config.data.dataset = DATASET
config.data.n_samples = 500
config.data.noise = 0.1
config.model.hidden_dims = HIDDEN_DIMS
config.train.epochs = EPOCHS
config.train.lr = 0.01
config.train.optimizer = "sgd"
config.train.momentum = 0.9
config.train.weight_decay = 0.0
config.train.scheduler = "none"
config.train.batch_size = 64
config.adv.epsilon = EPSILON
config.adv.step_size = EPSILON / 4
config.adv.num_steps = 7
config.adv.norm = "l2"
config.tess.num_checkpoints = 50
config.tess.checkpoint_epochs = _compute_checkpoint_epochs(EPOCHS, 50)
config.tess.local_complexity_radius = 0.1
config.seed = SEED
config.device = "cuda" if torch.cuda.is_available() else "cpu"

# Output directories (Colab-friendly)
config.checkpoint_dir = f"/content/checkpoints_{DATASET}"
config.figure_dir = f"/content/figures_{DATASET}"
config.results_dir = f"/content/results_{DATASET}"

os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.figure_dir, exist_ok=True)
os.makedirs(config.results_dir, exist_ok=True)

print(f"Dataset: {DATASET}")
print(f"Architecture: {HIDDEN_DIMS}")
print(f"Epochs: {EPOCHS}")
print(f"Epsilon: {EPSILON}")
print(f"Checkpoints: {len(config.tess.checkpoint_epochs)} evenly spaced")
print(f"Device: {config.device}")

## 7. Generate dataset

In [ ]:
dataset, X_train, y_train, X_test, y_test = get_dataset(
    config.data.dataset,
    n_samples=config.data.n_samples,
    noise=config.data.noise,
    seed=config.data.seed,
    n_test=200,
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train range: x=[{X_train[:,0].min():.3f}, {X_train[:,0].max():.3f}], "
      f"y=[{X_train[:,1].min():.3f}, {X_train[:,1].max():.3f}]")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_dataset(X_train, y_train, title=f"{DATASET} (train, n=500)", ax=axes[0])
plot_dataset(X_test, y_test, title=f"{DATASET} (test, n=200)", ax=axes[1])
plt.tight_layout()
plt.show()

## 8. Train models (Standard ERM + PGD-AT)

In [ ]:
def run_training(config, X_train, y_train, dataset, run_name):
    """Train one model and return history."""
    torch.manual_seed(config.seed)
    np.random.seed(config.seed)

    model = make_relu_mlp(
        input_dim=config.model.input_dim,
        hidden_dims=config.model.hidden_dims,
        output_dim=config.model.output_dim,
    )
    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"  Architecture: {config.model.hidden_dims}")
    print(f"  Parameters: {count_parameters(model)}")
    print(f"  Adversarial: {config.adv.enabled}")
    print(f"{'='*60}")

    dataloader = get_dataloader(dataset, batch_size=config.train.batch_size)
    checkpoint_dir = os.path.join(config.checkpoint_dir, run_name)

    history = train_model(
        model, dataloader, config,
        checkpoint_dir=checkpoint_dir,
        run_name=run_name,
        device=config.device,
    )
    return history, checkpoint_dir

In [ ]:
# --- Standard (ERM) training ---
config_std = copy.deepcopy(config)
config_std.adv.enabled = False

hist_std, ckpt_dir_std = run_training(
    config_std, X_train, y_train, dataset, "standard"
)

In [ ]:
# --- Adversarial (PGD-AT) training ---
config_adv = copy.deepcopy(config)
config_adv.adv.enabled = True

hist_adv, ckpt_dir_adv = run_training(
    config_adv, X_train, y_train, dataset, "adversarial"
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_std["epoch"], hist_std["train_loss"], label="Standard", color="#2196F3")
axes[0].plot(hist_adv["epoch"], hist_adv["train_loss"], label="Adversarial", color="#FF5722")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training Loss")
axes[0].set_title("Training Loss")
axes[0].legend()

axes[1].plot(hist_std["epoch"], hist_std["train_acc"], label="Standard", color="#2196F3")
axes[1].plot(hist_adv["epoch"], hist_adv["train_acc"], label="Adversarial", color="#FF5722")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Training Accuracy")
axes[1].set_title("Training Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Compute tessellation statistics at each checkpoint

This step loads each saved checkpoint, computes grid-based statistics (region count, local complexity, boundary distances, boundary density), the Poisson baseline, and — if SplineCam + CUDA are available — the exact SplineCam tessellation.

In [ ]:
def analyze_all_checkpoints(config, run_name, ckpt_dir, X_train):
    """Analyze all checkpoints for one training run."""
    model_factory = lambda: make_relu_mlp(
        input_dim=config.model.input_dim,
        hidden_dims=config.model.hidden_dims,
        output_dim=config.model.output_dim,
    )

    all_stats = []
    all_grids = []

    for epoch in config.tess.checkpoint_epochs:
        ckpt_path = os.path.join(ckpt_dir, f"{run_name}_epoch{epoch:04d}.pt")
        if not os.path.exists(ckpt_path):
            print(f"  Checkpoint not found: epoch {epoch}")
            continue

        print(f"  Analyzing epoch {epoch}...", end=" ")
        stats, grid_data = analyze_checkpoint(
            model_factory, ckpt_path, X_train, config, device=config.device
        )
        all_stats.append(stats)
        all_grids.append(grid_data)

        print(f"Regions={stats['num_regions_grid']} | "
              f"LC={stats.get('local_region_count_mean', 0):.1f} | "
              f"DBD={stats.get('boundary_dist_mean', 0):.4f} | "
              f"BD={stats.get('boundary_density', 0):.2f}")

    return all_stats, all_grids

In [ ]:
print("Analyzing STANDARD checkpoints...")
stats_std, grids_std = analyze_all_checkpoints(
    config_std, "standard", ckpt_dir_std, X_train
)

In [ ]:
print("Analyzing ADVERSARIAL checkpoints...")
stats_adv, grids_adv = analyze_all_checkpoints(
    config_adv, "adversarial", ckpt_dir_adv, X_train
)

## 10. Generate paper figures

In [ ]:
%matplotlib inline

# Main metrics comparison (Fig 1 equivalent)
plot_training_comparison(stats_std, stats_adv, figure_dir=config.figure_dir)

# Reload and display
from IPython.display import Image, display
display(Image(filename=os.path.join(config.figure_dir, "metrics_comparison.png")))

In [ ]:
# Epoch snapshot visualizations
snapshot_epochs = [s["epoch"] for s in stats_std]

# Pick a few representative epochs for visualization
vis_indices = [0, len(stats_std)//4, len(stats_std)//2, -1]
vis_stats_std = [stats_std[i] for i in vis_indices]
vis_grids_std = [grids_std[i] for i in vis_indices]
vis_epochs = [stats_std[i]["epoch"] for i in vis_indices]

plot_epoch_snapshots(
    vis_grids_std, X_train, y_train, vis_epochs,
    run_name="standard", figure_dir=config.figure_dir
)
display(Image(filename=os.path.join(config.figure_dir, "snapshots_standard.png")))

In [ ]:
vis_stats_adv = [stats_adv[i] for i in vis_indices]
vis_grids_adv = [grids_adv[i] for i in vis_indices]
vis_epochs_adv = [stats_adv[i]["epoch"] for i in vis_indices]

plot_epoch_snapshots(
    vis_grids_adv, X_train, y_train, vis_epochs_adv,
    run_name="adversarial", figure_dir=config.figure_dir
)
display(Image(filename=os.path.join(config.figure_dir, "snapshots_adversarial.png")))

In [ ]:
# Boundary distance histograms
plot_boundary_distance_histograms(
    stats_std, stats_adv, epoch_idx=-1, figure_dir=config.figure_dir
)
ep = stats_std[-1]["epoch"]
display(Image(filename=os.path.join(config.figure_dir, f"boundary_dist_hist_ep{ep}.png")))

## 11. Poisson baseline comparison

In [ ]:
# Compare learned tessellation vs matched-intensity Poisson baseline
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs_s = [s["epoch"] for s in stats_std]
epochs_a = [s["epoch"] for s in stats_adv]

# (a) Region count: learned vs Poisson
ax = axes[0]
ax.plot(epochs_s, [s["num_regions_grid"] for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(epochs_a, [s["num_regions_grid"] for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.plot(epochs_s, [s.get("poisson_num_regions", 0) for s in stats_std],
        '--', label="Poisson (std)", color='#2196F3', alpha=0.5)
ax.plot(epochs_a, [s.get("poisson_num_regions", 0) for s in stats_adv],
        '--', label="Poisson (adv)", color='#FF5722', alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("# Regions")
ax.set_title("Region Count: Learned vs Poisson")
ax.legend(fontsize=8)

# (b) Cell area median
ax = axes[1]
ax.plot(epochs_s, [s.get("cell_area_median", 0) for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(epochs_a, [s.get("cell_area_median", 0) for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.plot(epochs_s, [s.get("poisson_cell_area_median", 0) for s in stats_std],
        '--', label="Poisson (std)", color='#2196F3', alpha=0.5)
ax.plot(epochs_a, [s.get("poisson_cell_area_median", 0) for s in stats_adv],
        '--', label="Poisson (adv)", color='#FF5722', alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Median Cell Area")
ax.set_title("Cell Area: Learned vs Poisson")
ax.legend(fontsize=8)

# (c) Boundary density
ax = axes[2]
ax.plot(epochs_s, [s.get("boundary_density", 0) for s in stats_std],
        'o-', label="Standard", color='#2196F3', markersize=3)
ax.plot(epochs_a, [s.get("boundary_density", 0) for s in stats_adv],
        's-', label="Adversarial", color='#FF5722', markersize=3)
ax.plot(epochs_s, [s.get("poisson_boundary_density", 0) for s in stats_std],
        '--', label="Poisson (std)", color='#2196F3', alpha=0.5)
ax.plot(epochs_a, [s.get("poisson_boundary_density", 0) for s in stats_adv],
        '--', label="Poisson (adv)", color='#FF5722', alpha=0.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("Boundary Density")
ax.set_title("Boundary Density: Learned vs Poisson")
ax.legend(fontsize=8)

plt.suptitle(f"Poisson Baseline Comparison ({DATASET})", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(config.figure_dir, "poisson_comparison.png"), dpi=200)
plt.savefig(os.path.join(config.figure_dir, "poisson_comparison.pdf"))
plt.show()

## 12. Save results

In [ ]:
def serialize_stats(stats_list):
    """Convert stats to JSON-serializable format."""
    out = []
    for s in stats_list:
        d = {}
        for k, v in s.items():
            if isinstance(v, np.ndarray):
                d[k] = v.tolist()
            elif isinstance(v, (np.float32, np.float64)):
                d[k] = float(v)
            elif isinstance(v, (np.int32, np.int64)):
                d[k] = int(v)
            else:
                try:
                    json.dumps(v)
                    d[k] = v
                except (TypeError, ValueError):
                    pass
        out.append(d)
    return out

results = {
    "config": {
        "dataset": DATASET,
        "hidden_dims": HIDDEN_DIMS,
        "epochs": EPOCHS,
        "epsilon": EPSILON,
        "lr": config.train.lr,
        "optimizer": config.train.optimizer,
        "momentum": config.train.momentum,
        "norm": config.adv.norm,
        "pgd_steps": config.adv.num_steps,
        "n_train": len(X_train),
        "n_test": len(X_test),
    },
    "standard": serialize_stats(stats_std),
    "adversarial": serialize_stats(stats_adv),
}

results_path = os.path.join(config.results_dir, "results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {results_path}")
print(f"Figures saved to {config.figure_dir}/")

In [ ]:
# Download results (optional)
from google.colab import files

# Zip everything
!cd /content && zip -r experiment_results.zip \
    figures_{DATASET}/ results_{DATASET}/ checkpoints_{DATASET}/ \
    -x '*.pyc' '__pycache__/*'

files.download("/content/experiment_results.zip")

## 13. Run additional architecture configurations

The paper sweeps over architectures with 1-2 hidden layers of width $w \in \{10, 20, 50\}$. Uncomment and run the cell below to sweep.

In [ ]:
# # Architecture sweep
# ARCHITECTURES = [
#     [10],       # 1 hidden layer, width 10
#     [20],       # 1 hidden layer, width 20
#     [50],       # 1 hidden layer, width 50
#     [10, 10],   # 2 hidden layers, width 10
#     [20, 20],   # 2 hidden layers, width 20
#     [50, 50],   # 2 hidden layers, width 50
# ]
#
# all_results = {}
# for arch in ARCHITECTURES:
#     arch_name = "x".join(map(str, arch))
#     print(f"\n{'#'*60}")
#     print(f"Architecture: {arch}")
#     print(f"{'#'*60}")
#
#     config.model.hidden_dims = arch
#     config_s = copy.deepcopy(config)
#     config_s.adv.enabled = False
#     config_a = copy.deepcopy(config)
#     config_a.adv.enabled = True
#
#     h_s, cd_s = run_training(config_s, X_train, y_train, dataset, f"std_{arch_name}")
#     h_a, cd_a = run_training(config_a, X_train, y_train, dataset, f"adv_{arch_name}")
#
#     s_s, g_s = analyze_all_checkpoints(config_s, f"std_{arch_name}", cd_s, X_train)
#     s_a, g_a = analyze_all_checkpoints(config_a, f"adv_{arch_name}", cd_a, X_train)
#
#     all_results[arch_name] = {
#         "standard": serialize_stats(s_s),
#         "adversarial": serialize_stats(s_a),
#     }
#
# # Save sweep results
# with open(os.path.join(config.results_dir, "sweep_results.json"), "w") as f:
#     json.dump(all_results, f, indent=2)
# print("Architecture sweep complete.")